In [0]:
%sql
select product_category_name, count(product_id) as qt from tmw_olist.products
group by all 
;

select count(product_id) as qt from tmw_olist.products
group by all 


In [0]:
%sql
select 
    -- count(order_id) as qt_orders,
    -- sum(price) as preco,
    -- sum(freight_value) as frete

    *
    
     from workspace.tmw_olist.order_items
group by all 


In [0]:
%sql
select count(seller_id) as qt_vendedores from workspace.tmw_olist.sellers
group by all 


In [0]:
%sql
select  count(geolocation_zip_code_prefix) as qt_ceps from tmw_olist.geolocation
group by all 


In [0]:
%sql
select count(customer_id) as clientes from workspace.tmw_olist.customers
group by all 



In [0]:
%sql
select /*payment_installments*/
count(order_id) as qt_pagto,sum(payment_value) as valor from workspace.tmw_olist.order_payments
group by all 



# order de pagamento sem pedido aberto ?


In [0]:

%sql
select 
  a.order_id,
  b.product_id
  -- b.customer_id

   from  tmw_olist.order_payments a 

left join tmw_olist.order_items b
-- left join tmw_olist.orders b
  on a.order_id=b.order_id 

where b.product_id is null
-- where b.customer_id is null


;

%sql
select * from tmw_olist.orders
where 
order_id = 'f86d7bc39aab05299691322044b63bb2'


In [0]:
%sql
select count(*) from tmw_olist.order_reviews
-- where review_answer_timestamp is null 

group by all 


In [0]:
%sql
select count(order_id) as qt_pedidos  from workspace.tmw_olist.orders
group by all 


In [0]:
%sql
select * from tmw_olist.product_category_name_translation

Existe pedidos cancelados com pagamento ?


In [0]:
%sql
select 
  a.order_id,
  a.order_purchase_timestamp,
  a.order_status,
  b.payment_sequential,
  b.payment_type,
  b.payment_installments,
  b.payment_value,
  a.*

 from tmw_olist.orders a

  left join tmw_olist.order_payments b 
    on a.order_id=b.order_id 
    
  where a.order_status = 'canceled' 
order by a.order_id 


existem analise do cliente sem pedido ?

In [0]:
%sql
select * from tmw_olist.order_reviews a 

left join tmw_olist.orders b 
  on a.order_id=b.order_id

where b.customer_id is null   

ABT OLIST


In [0]:
data_fim = '2025-01-01'

In [0]:
spark.sql(f"""
        -- create or replace view vw_pedidos  as 
        with pedidos as (

        select 
            a.order_status,
            cast(date_format(a.order_purchase_timestamp, "yyyy-MM-dd") as date )as Dt_Pedido , 
            cast(date_format(date_trunc("MONTH",a.order_purchase_timestamp), "yyyy-MM-dd") as date )as AnoMes_Pedido , 
            cast(date_format(a.order_approved_at, "yyyy-MM-dd") as date )as Dt_Aprovado, 
            cast(date_format(a.order_delivered_carrier_date, "yyyy-MM-dd") as date )as Dt_Transportadora, 
            cast(date_format(a.order_delivered_customer_date, "yyyy-MM-dd") as date )as Dt_Entregue, 
            cast(date_format(a.order_estimated_delivery_date, "yyyy-MM-dd") as date )as Dt_Prazo, 
            b.order_item_id as ordem_itens,
            b.price as preco,
            b.freight_value as frete,
            c.product_category_name as categoria,
            c.product_name_lenght,
            c.product_description_lenght,
            c.product_photos_qty,
            c.product_weight_g,
            c.product_length_cm,
            c.product_height_cm,
            c.product_width_cm,
            d.payment_sequential as seq_pagto,
            d.payment_type as meio_pagamento,
            d.payment_installments as parcelas,
            d.payment_value as valor, 
            e.customer_zip_code_prefix as cep_cliente,
            e.customer_city as cidade_cliente,
            e.customer_state as estado_cliente,
            f.seller_zip_code_prefix as cep_vendedor,
            f.seller_city as cidade_vendedor,
            f.seller_state as estado_vendedor,
            g.review_score,
            g.review_comment_title,
            g.review_comment_message,
            cast(date_format(g.review_creation_date, "yyyy-MM-dd") as date )as Dt_Nota_Cliente, 
            cast(date_format(g.review_answer_timestamp, "yyyy-MM-dd") as date )as Dt_Resposta_Nota
          
            from tmw_olist.orders a

            left join tmw_olist.order_items b 
                on a.order_id = b.order_id 

            left join tmw_olist.products c 
                on b.product_id = c.product_id 

             left join tmw_olist.order_payments d 
                 on a.order_id = d.order_id 

             left join tmw_olist.customers e 
                 on a.customer_id = e.customer_id 

             left join tmw_olist.sellers f 
                 on b.seller_id = f.seller_id

             left join tmw_olist.order_reviews g 
                 on a.order_id = g.order_id 
               

        where a.order_purchase_timestamp < '{data_fim}'
        and
        --  a.order_id = '00526a9d4ebde463baee25f386963ddc'
         e.customer_id = '1f1c7bf1c9b041b292af6c1c4470b753'
        )

        select * from pedidos 
""").display() 






In [0]:
%sql
-- select * from workspace.tmw_olist.customers

select * from tmw_olist.orders
where customer_id = '1f1c7bf1c9b041b292af6c1c4470b753'
